# 01. Pobranie i kontrola danych Binance

Pobieramy dzienne świece BNB/USDT i BTC/USDT od 15.06.2022 (co najmniej 200 dni przed analizą), do ostatniego zamkniętego dnia UTC. Surowe dane pozostają w pamięci. Wyniki kontroli nie zawierają pełnej historii notowań. Uruchom notebook od początku do końca; potrzebne jest połączenie z internetem.

In [1]:
from market_data import download
import pandas as pd

now = pd.Timestamp.now(tz="UTC")
data = {}
checks = {}
for symbol in ("BNBUSDT", "BTCUSDT"):
    data[symbol], checks[symbol] = download(symbol, now=now)
    c = checks[symbol]
    print(f"{symbol}: {len(data[symbol])} poprawnych świec; ostatni oczekiwany dzień UTC: {c['as_of'].date()}; ostatni dostępny: {data[symbol].index.max().date()}")
    print(f"  niepoprawne: {c['invalid']}; rekordy o zdublowanych datach: {c['duplicate_rows']}; dane poza kolejnością: {c['unordered']}")
    print(f"  brakujące dni ({len(c['missing_dates'])}): {[str(d.date()) for d in c['missing_dates']]}")
    print("  dzienne zmiany ceny powyżej 50% do ręcznej weryfikacji:")
    print(c['outliers'].to_string() if len(c['outliers']) else "  brak")

BNBUSDT: 1615 poprawnych świec; ostatni oczekiwany dzień UTC: 2026-11-15; ostatni dostępny: 2026-11-15
  niepoprawne: 0; rekordy o zdublowanych datach: 0; dane poza kolejnością: False
  brakujące dni (0): []
  dzienne zmiany ceny powyżej 50% do ręcznej weryfikacji:
  brak


BTCUSDT: 1615 poprawnych świec; ostatni oczekiwany dzień UTC: 2026-11-15; ostatni dostępny: 2026-11-15
  niepoprawne: 0; rekordy o zdublowanych datach: 0; dane poza kolejnością: False
  brakujące dni (0): []
  dzienne zmiany ceny powyżej 50% do ręcznej weryfikacji:
  brak


Walidacja odrzuca błędne rekordy oraz wszystkie rekordy z powtórzoną datą, sortuje dane i zgłasza luki. Dzisiejsza świeca UTC jest wykluczona. Zwroty obejmujące lukę nie są traktowane jako dzienne; duże zmiany nie są usuwane.